# HyperDream Colab 猛版一键实验（GitHub 直连）

这份 Notebook 给你一个目标：**刷新 Colab 后，尽量少改，直接开跑**。

```text
流程总览
├─ 1) 拉最新代码（自动处理 ff-only 冲突）
├─ 2) 安装依赖
├─ 3) 设置 aggressive 参数
├─ 4) 从 Colab Secrets 读取 GitHub Token
├─ 5) 先做 GitHub 推送权限预检（5 秒内知道能不能 push）
├─ 6) 一键跑测试 + 四类实验 + 自动回传
└─ 7) 打印结果分支链接
```


## Cell 1：拉代码（首次 clone，之后自动 pull）

大白话：
- 你每次刷新 Colab，这一步都可以重复执行。
- 如果本地状态乱了，脚本会自动对齐到 `origin/main`，避免卡在 `pull --ff-only`。


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/peter941221/High_Dimensional_WorldModel.git'
PROJECT_DIR = Path('/content/High_Dimensional_WorldModel')
BRANCH = 'main'

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    try:
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    except subprocess.CalledProcessError:
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)

    pull = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH],
        text=True,
        capture_output=True,
    )
    if pull.returncode != 0:
        if pull.stdout:
            print(pull.stdout)
        if pull.stderr:
            print(pull.stderr)
        print('检测到 ff-only 冲突，自动 reset 到 origin/main')
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

os.chdir(PROJECT_DIR)
print('当前目录:', Path.cwd())


## Cell 2：安装依赖

大白话：
- 这一步就是把运行训练所需的 Python 包装好。
- 不报错，后面才会稳。


In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt


## Cell 3：实验参数（猛版）

大白话：
- 这套参数比默认大很多，训练更久、评估更稳。
- 如果你今天想先快测，把这些数字调小就行。


In [ ]:
from datetime import datetime

RUN_ID = f'colab_aggressive_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
RESUME = False

BASELINE_EPOCHS = 12
TRANSFER_PRETRAIN_EPOCHS = 8
TRANSFER_FINETUNE_EPOCHS = 8
ABLATION_EPOCHS = 8
ROBUSTNESS_EPISODES = 120
EVAL_EPISODES = 40
MAX_STEPS = 120

SAVE_EVERY = 4
KEEP_LAST = 6

RUN_TESTS = True
PUSH_RESULTS_TO_GITHUB = True
PUSH_BRANCH = 'colab-results'
GITHUB_USER = 'peter941221'
REPO_NAME = 'High_Dimensional_WorldModel'
SECRET_NAME = 'GITHUB_TOKEN'
TOKEN_ENV = 'GITHUB_TOKEN'

print('RUN_ID =', RUN_ID)
print('RESUME =', RESUME)


## Cell 4：读取 Colab Secrets（Token）

大白话：
- 不手输 token，直接从 Secrets 读。
- 会自动尝试多个常见键名，尽量减少你改配置。


In [ ]:
import os
from google.colab import userdata

if PUSH_RESULTS_TO_GITHUB:
    token = None
    candidates = [SECRET_NAME, 'GITHUB_TOKEN', 'GITHUB_1', 'GITHUB_T', 'GH_TOKEN']
    seen = set()
    candidates = [c for c in candidates if c and not (c in seen or seen.add(c))]
    for key in candidates:
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            token = str(value).strip()
            print(f'读取到 Secret: {key}')
            break
    if not token:
        raise ValueError(f'在 Colab Secrets 未找到可用 token，已尝试: {candidates}')
    os.environ[TOKEN_ENV] = token
    print(f'已写入环境变量: {TOKEN_ENV}')
else:
    print('跳过 token 读取（PUSH_RESULTS_TO_GITHUB=False）')


## Cell 5：先做 push 权限预检（强烈建议）

大白话：
- 这一步 5 秒内就知道：token 是否真的有写仓库权限。
- 通过再训练，就不会出现“训练20分钟后最后一步才403”。


In [ ]:
import subprocess

if PUSH_RESULTS_TO_GITHUB:
    preflight_cmd = [
        'python', 'colab_push_results.py',
        '--repo-dir', '/content/High_Dimensional_WorldModel',
        '--run-id', 'preflight_probe',
        '--branch', PUSH_BRANCH,
        '--github-user', GITHUB_USER,
        '--repo-name', REPO_NAME,
        '--token-env', TOKEN_ENV,
        '--token-secret-name', SECRET_NAME,
        '--check-push-access-only',
    ]
    print('预检命令:')
    print(' '.join(preflight_cmd))
    subprocess.run(preflight_cmd, check=True)
    print('Push 权限预检通过')
else:
    print('跳过 push 权限预检')


## Cell 6：一键跑（测试 + 训练 + 出图 + 自动 push）

大白话：
- 这是主命令。
- 执行后会按顺序跑 baseline / transfer / ablation / robustness，再画图并推送到 GitHub。
- 失败时会打印完整 stdout/stderr，方便定位。


In [ ]:
import subprocess

cmd = [
    'python', 'colab_autorun.py',
    '--run-id', RUN_ID,
    '--baseline-epochs', str(BASELINE_EPOCHS),
    '--transfer-pretrain-epochs', str(TRANSFER_PRETRAIN_EPOCHS),
    '--transfer-finetune-epochs', str(TRANSFER_FINETUNE_EPOCHS),
    '--ablation-epochs', str(ABLATION_EPOCHS),
    '--robustness-episodes', str(ROBUSTNESS_EPISODES),
    '--eval-episodes', str(EVAL_EPISODES),
    '--max-steps', str(MAX_STEPS),
    '--save-every', str(SAVE_EVERY),
    '--keep-last', str(KEEP_LAST),
]

if RESUME:
    cmd.append('--resume')
if RUN_TESTS:
    cmd.append('--run-tests')

if PUSH_RESULTS_TO_GITHUB:
    cmd.extend([
        '--push-results-to-github',
        '--push-branch', PUSH_BRANCH,
        '--github-user', GITHUB_USER,
        '--repo-name', REPO_NAME,
        '--token-env', TOKEN_ENV,
        '--token-secret-name', SECRET_NAME,
    ])

print('将执行命令:')
print(' '.join(cmd))
proc = subprocess.run(cmd, text=True, capture_output=True)
print(f'RETURN CODE: {proc.returncode}')
if proc.stdout:
    print('==== STDOUT ====')
    print(proc.stdout)
if proc.stderr:
    print('==== STDERR ====')
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError('colab_autorun failed; 真实错误已打印在上面')
print('训练与回传完成')


## Cell 7：查看结果链接

大白话：
- 直接打开这个分支链接看结果文件和图。


In [ ]:
!git branch -a
!git log --oneline -n 8
print(f'结果分支: https://github.com/{GITHUB_USER}/{REPO_NAME}/tree/{PUSH_BRANCH}')
print(f'本次 RUN_ID: {RUN_ID}')
